In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Silver

In [0]:
log("Reading from Silver products ...")
df_silver_products = spark.table(TBL_SILVER_PRODUCTS)
df_silver_products.show()

## 2. Build dim_product

In [0]:
df_dim_product = df_silver_products.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("sub_category")
)

## 3. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_DIM_PRODUCT):
    log(f"Upserting into {TBL_GOLD_DIM_PRODUCT} ...")

    delta_table = DeltaTable.forName(spark, TBL_GOLD_DIM_PRODUCT)
    (
        delta_table.alias("target")
        .merge(
            df_dim_product.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_PRODUCT} upserted")

else:
    log(f"Creating {TBL_GOLD_DIM_PRODUCT}")
    (
        df_dim_product.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TBL_GOLD_DIM_PRODUCT)
    )
    
    log(f"✅ Done. Table {TBL_GOLD_DIM_PRODUCT} created")

In [0]:
display(spark.table(TBL_GOLD_DIM_PRODUCT))